In [7]:
import pandas as pd
device = pd.read_csv("/Users/dholakiyan/Desktop/insider-threat-detection/data/raw/r4.2/device.csv")
device['date'] = pd.to_datetime(device['date'])

print(device.shape)
print(device.head())
print(device['activity'].value_counts())

(405380, 5)
                         id                date     user       pc    activity
0  {J1S3-L9UU75BQ-7790ATPL} 2010-01-02 07:21:06  MOH0273  PC-6699     Connect
1  {N7B5-Y7BB27SI-2946PUJK} 2010-01-02 07:37:41  MOH0273  PC-6699  Disconnect
2  {U1V9-Z7XT67KV-5649MYHI} 2010-01-02 07:59:11  HPH0075  PC-2417     Connect
3  {H0Z7-E6GB57XZ-1603MOXD} 2010-01-02 07:59:49  IIW0249  PC-0843     Connect
4  {L7P2-G4PX02RX-7999GYOY} 2010-01-02 08:04:26  IIW0249  PC-0843  Disconnect
activity
Connect       203339
Disconnect    202041
Name: count, dtype: int64


### Filter to your suspect user, across the full timeline

In [8]:
suspect_device = device[device['user'] == 'AAM0658'].sort_values('date')
print(suspect_device)

                              id                date     user       pc  \
253170  {K9S3-D9LS33CJ-8715FSIC} 2010-10-21 00:22:06  AAM0658  PC-9923   
253171  {R5T4-K9IN57VQ-3316ULRN} 2010-10-21 00:38:02  AAM0658  PC-9923   
253172  {B1K2-L6TJ01DP-1156PAZE} 2010-10-21 00:44:55  AAM0658  PC-9923   
253173  {G1K3-A7RJ76HE-1102PRZE} 2010-10-21 01:03:23  AAM0658  PC-9923   
253174  {Y0Z3-D7MK17DG-9916IXDE} 2010-10-21 01:08:13  AAM0658  PC-9923   
253175  {V5K4-B2PS79AF-7599SFAP} 2010-10-21 01:18:53  AAM0658  PC-9923   
255349  {H1L0-X7RH83FI-5967VUQY} 2010-10-23 06:18:48  AAM0658  PC-9923   
255350  {W7C0-G1SW41KB-1991BUTL} 2010-10-23 06:26:48  AAM0658  PC-9923   
257685  {G0R8-K2YH87AK-9571JLRM} 2010-10-27 04:37:05  AAM0658  PC-9923   
257688  {W0X8-S1EX02AJ-2043CZDR} 2010-10-27 04:59:04  AAM0658  PC-9923   
259814  {S1J0-Z2ZX87IX-2607ZUXY} 2010-10-29 01:30:10  AAM0658  PC-9923   
259827  {V6D8-B3FZ26HT-3583PJAI} 2010-10-29 05:00:19  AAM0658  PC-9923   
262213  {U4Q1-R8JZ85OJ-7363YIPP} 2010-

### Check specifically around the malicious window

In [9]:
window_start = pd.Timestamp("2010-10-20")
window_end = pd.Timestamp("2010-11-03")

suspect_device_window = suspect_device[
    (suspect_device['date'] >= window_start) & 
    (suspect_device['date'] <= window_end)
]
print(suspect_device_window)

                              id                date     user       pc  \
253170  {K9S3-D9LS33CJ-8715FSIC} 2010-10-21 00:22:06  AAM0658  PC-9923   
253171  {R5T4-K9IN57VQ-3316ULRN} 2010-10-21 00:38:02  AAM0658  PC-9923   
253172  {B1K2-L6TJ01DP-1156PAZE} 2010-10-21 00:44:55  AAM0658  PC-9923   
253173  {G1K3-A7RJ76HE-1102PRZE} 2010-10-21 01:03:23  AAM0658  PC-9923   
253174  {Y0Z3-D7MK17DG-9916IXDE} 2010-10-21 01:08:13  AAM0658  PC-9923   
253175  {V5K4-B2PS79AF-7599SFAP} 2010-10-21 01:18:53  AAM0658  PC-9923   
255349  {H1L0-X7RH83FI-5967VUQY} 2010-10-23 06:18:48  AAM0658  PC-9923   
255350  {W7C0-G1SW41KB-1991BUTL} 2010-10-23 06:26:48  AAM0658  PC-9923   
257685  {G0R8-K2YH87AK-9571JLRM} 2010-10-27 04:37:05  AAM0658  PC-9923   
257688  {W0X8-S1EX02AJ-2043CZDR} 2010-10-27 04:59:04  AAM0658  PC-9923   
259814  {S1J0-Z2ZX87IX-2607ZUXY} 2010-10-29 01:30:10  AAM0658  PC-9923   
259827  {V6D8-B3FZ26HT-3583PJAI} 2010-10-29 05:00:19  AAM0658  PC-9923   
262213  {U4Q1-R8JZ85OJ-7363YIPP} 2010-

### Compare to their USB usage BEFORE the window

In [10]:
before_window = suspect_device[suspect_device['date'] < window_start]
print("USB events before Oct 20:", before_window.shape[0])
print("USB events during window:", suspect_device_window.shape[0])

USB events before Oct 20: 0
USB events during window: 13


### Confirmed Multi-Source Detection: User AAM0658 (Scenario 1)

Cross-referencing `logon.csv` and `device.csv` reveals a complete, 
correlated behavioral signature:

| Date | Logon (after-hours) | USB Activity |
|---|---|---|
| Oct 21 | 00:18–01:20 AM | 3x Connect/Disconnect cycles (00:22–01:18) |
| Oct 23 | 01:34 AM | Connect/Disconnect (06:18–06:26) |
| Oct 27 | 04:31–05:54 AM | Connect/Disconnect (04:37–04:59) |
| Oct 29 | 00:06–05:23 AM | Connect/Disconnect (01:30–05:00) |
| Nov 2 | 02:31–03:02 AM | Connect (02:40) |

Zero USB events exist anywhere in this user's history before Oct 20, 2010. 
The tight time correlation between after-hours logons and USB connect/
disconnect cycles across 5 separate nights strongly matches Scenario 1's 
described behavior: an employee with no prior removable-media usage who 
begins working after hours and using a USB drive shortly before departure.

This confirms the injected threat signal is detectable through combined 
logon + device behavioral analysis, validating the project's anomaly 
detection approach.